# 03 — Scoring, ADMET-AI, similitud estructural y selección Pareto de candidatas EGFR

Este notebook constituye la etapa de **priorización post-generativa**. Parte de las moléculas generadas en el notebook anterior y aplica una secuencia de filtros y rankings para obtener una lista final de candidatas.

El objetivo final es exportar un CSV con las **30 mejores candidatas**, que será utilizado en un notebook posterior para análisis e interpretación más profunda.

## Objetivo operativo

1. Importar moléculas generadas y eliminar entradas no válidas o presentes en el conjunto de entrenamiento.
2. Calcular descriptores fisicoquímicos con RDKit.
3. Entrenar un modelo QSAR de pIC50 sobre el dataset EGFR de referencia.
4. Predecir actividad de las moléculas generadas.
5. Evaluar similitud frente al conjunto de entrenamiento para controlar novedad y dominio de aplicabilidad.
6. Integrar predicciones ADMET-AI cuando estén disponibles.
7. Construir scores interpretables de actividad, ADMET, similitud, QED y Lipinski.
8. Aplicar selección Pareto y control de diversidad por scaffold.
9. Exportar el CSV final con el top 30 de candidatas.

> Nota metodológica: este notebook no valida experimentalmente las moléculas. Su función es priorizar candidatos razonables para análisis posterior. Las predicciones deben interpretarse como criterios de cribado computacional, no como evidencia biológica definitiva.


## 1. Configuración del entorno y rutas

Se centralizan las rutas y parámetros principales para que el notebook sea reproducible. Las entradas se leen desde `data/` y `outputs/mrl/`, mientras que todas las salidas de este notebook se guardan bajo `outputs/scoring/`, separadas en `intermediate/`, `models/`, `final/` y `reports/`.

Esta separación evita mezclar las salidas del modelo generativo con las salidas del proceso de scoring y selección.

In [2]:
from pathlib import Path
from datetime import datetime
import json
import sys
import warnings

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)

# ---------------------------------------------------------------------
# Rutas del proyecto
# ---------------------------------------------------------------------

# Carpeta desde la que se está ejecutando el notebook.
NOTEBOOK_DIR = Path.cwd().resolve()

# Raíz del proyecto; si el notebook se ejecuta desde notebooks/ o src/, se sube un nivel.
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() in {"notebooks", "src"} else NOTEBOOK_DIR

# Carpeta donde están los datasets curados y referencias externas del proyecto.
DATA_DIR = PROJECT_ROOT / "data"

# Carpeta principal donde se guardan todos los resultados generados por los notebooks.
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# Carpeta con las salidas del modelo generativo MRL del notebook 02.
MRL_OUTPUT_DIR = OUTPUTS_DIR / "mrl"

# Carpeta raíz de las salidas específicas de este notebook de scoring.
SCORING_DIR = OUTPUTS_DIR / "scoring"

# Carpeta para archivos intermedios del scoring; permite auditar cada etapa sin recalcular todo.
SCORING_INTERMEDIATE_DIR = SCORING_DIR / "intermediate"

# Carpeta para modelos entrenados en este notebook y sus métricas asociadas.
SCORING_MODELS_DIR = SCORING_DIR / "models"

# Carpeta para salidas finales que serán consumidas por el notebook 04.
SCORING_FINAL_DIR = SCORING_DIR / "final"

# Carpeta para resúmenes de ejecución legibles y trazables.
SCORING_REPORTS_DIR = SCORING_DIR / "reports"

for directory in [SCORING_INTERMEDIATE_DIR, SCORING_MODELS_DIR, SCORING_FINAL_DIR, SCORING_REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Archivo de moléculas generadas por el modelo MRL; entrada principal de este notebook.
GENERATED_PATH = MRL_OUTPUT_DIR / "generated_molecules.csv"

# Dataset procesado de actividad EGFR usado como base para entrenar el predictor de pIC50.
TRAINING_DATA_PATH = DATA_DIR / "model_procesed_dataset.csv"

# Dataset preprocesado de referencia con SMILES y pIC50 para modelado y validación.
REFERENCE_DATA_PATH = DATA_DIR / "model_preprocesed_dataset.csv"

# Archivo opcional con fármacos EGFR aprobados o de referencia para comparación estructural.
APPROVED_EGFR_REF_PATH = DATA_DIR / "fda_ema_egfr_reference.csv"

# Archivo intermedio con las moléculas generadas y sus descriptores fisicoquímicos RDKit.
DESCRIPTORS_OUTPUT_PATH = SCORING_INTERMEDIATE_DIR / "01_generated_molecules_descriptors.csv"

# Archivo intermedio con las moléculas que superan el filtro químico básico.
FILTERED_OUTPUT_PATH = SCORING_INTERMEDIATE_DIR / "02_generated_molecules_filtered.csv"

# Archivo intermedio preparado como entrada para ADMET-AI.
ADMET_INPUT_PATH = SCORING_INTERMEDIATE_DIR / "03_admet_ai_input.csv"

# Archivo intermedio con las predicciones devueltas por ADMET-AI.
ADMET_OUTPUT_PATH = SCORING_INTERMEDIATE_DIR / "04_admet_ai_predictions.csv"

# Archivo intermedio con todas las moléculas filtradas y sus scores integrados.
SCORED_OUTPUT_PATH = SCORING_INTERMEDIATE_DIR / "05_generated_molecules_scored.csv"

# Archivo intermedio con moléculas que pasan los criterios mínimos de elegibilidad.
ELIGIBLE_OUTPUT_PATH = SCORING_INTERMEDIATE_DIR / "06_eligible_candidates.csv"

# Archivo intermedio con candidatas pertenecientes al frente Pareto.
PARETO_OUTPUT_PATH = SCORING_INTERMEDIATE_DIR / "07_pareto_candidates.csv"

# Modelo Random Forest entrenado para predecir pIC50 frente a EGFR.
ACTIVITY_MODEL_PATH = SCORING_MODELS_DIR / "egfr_pic50_random_forest.pkl"

# Métricas del modelo predictivo de pIC50 para justificar su rendimiento.
ACTIVITY_METRICS_PATH = SCORING_MODELS_DIR / "egfr_pic50_random_forest_metrics.json"

# Archivo final con hasta 100 candidatas priorizadas y diversificadas por scaffold.
TOP100_OUTPUT_PATH = SCORING_FINAL_DIR / "top100_pareto_diverse.csv"

# Archivo final principal: top 30 candidatas para análisis y conclusiones posteriores.
SHORTLIST_OUTPUT_PATH = SCORING_FINAL_DIR / "shortlist30_final_candidates.csv"

# Resumen estructurado de la ejecución, útil para trazabilidad y lectura desde otros notebooks.
SCORING_SUMMARY_JSON_PATH = SCORING_REPORTS_DIR / "scoring_summary.json"

# Resumen en texto plano de la ejecución, útil para revisar rápidamente resultados y criterios.
SCORING_SUMMARY_TXT_PATH = SCORING_REPORTS_DIR / "scoring_summary.txt"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GENERATED_PATH:", GENERATED_PATH)
print("TRAINING_DATA_PATH:", TRAINING_DATA_PATH)
print("REFERENCE_DATA_PATH:", REFERENCE_DATA_PATH)
print("SCORING_DIR:", SCORING_DIR)


PROJECT_ROOT: /home/kluna/TFM/egfr_pipeline_tfm
GENERATED_PATH: /home/kluna/TFM/egfr_pipeline_tfm/outputs/mrl/generated_molecules.csv
TRAINING_DATA_PATH: /home/kluna/TFM/egfr_pipeline_tfm/data/model_procesed_dataset.csv
REFERENCE_DATA_PATH: /home/kluna/TFM/egfr_pipeline_tfm/data/model_preprocesed_dataset.csv
SCORING_DIR: /home/kluna/TFM/egfr_pipeline_tfm/outputs/scoring


## 2. Dependencias químicas y funciones auxiliares

RDKit se utiliza para:

- validar y canonizar SMILES,
- calcular descriptores fisicoquímicos,
- obtener fingerprints Morgan,
- calcular scaffolds de Murcko,
- medir similitud Tanimoto.


In [3]:
try:
    from rdkit import Chem, DataStructs
    from rdkit.Chem import Descriptors, Crippen, Lipinski, QED, rdMolDescriptors
    from rdkit.Chem.Scaffolds import MurckoScaffold
    from rdkit.Chem import rdFingerprintGenerator
except ImportError as e:
    raise ImportError("RDKit no está disponible. Instálalo antes de ejecutar este notebook.") from e

try:
    import joblib
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
except ImportError as e:
    raise ImportError("Faltan dependencias de scikit-learn/joblib.") from e

try:
    from scipy.stats import spearmanr
except ImportError:
    spearmanr = None


def mol_from_smiles(smiles: str):
    """Convierte un SMILES en molécula RDKit. Devuelve None si no es válido."""
    if pd.isna(smiles):
        return None
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        return mol
    except Exception:
        return None


def canonicalize_smiles(smiles: str):
    """Canoniza SMILES para comparar moléculas de forma consistente."""
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)


def ro5_violations_from_mol(mol) -> int:
    """Cuenta violaciones básicas de la regla de Lipinski."""
    return (
        int(Descriptors.MolWt(mol) > 500)
        + int(Crippen.MolLogP(mol) > 5)
        + int(Lipinski.NumHDonors(mol) > 5)
        + int(Lipinski.NumHAcceptors(mol) > 10)
    )


def get_murcko_scaffold(smiles: str):
    """Extrae el scaffold de Murcko como SMILES canónico."""
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    if scaffold is None or scaffold.GetNumAtoms() == 0:
        return ""
    return Chem.MolToSmiles(scaffold, canonical=True)


def calc_basic_descriptors(smiles: str) -> pd.Series:
    """Calcula descriptores RDKit utilizados en filtros y ranking."""
    mol = mol_from_smiles(smiles)
    if mol is None:
        return pd.Series(dtype="float64")
    return pd.Series({
        "canonical_smiles": Chem.MolToSmiles(mol, canonical=True),
        "mol_wt": Descriptors.MolWt(mol),
        "rdkit_logp": Crippen.MolLogP(mol),
        "hbd": Lipinski.NumHDonors(mol),
        "hba": Lipinski.NumHAcceptors(mol),
        "tpsa": rdMolDescriptors.CalcTPSA(mol),
        "rot_bonds": Lipinski.NumRotatableBonds(mol),
        "rings": rdMolDescriptors.CalcNumRings(mol),
        "heavy_atoms": mol.GetNumHeavyAtoms(),
        "qed": QED.qed(mol),
        "ro5_violations": ro5_violations_from_mol(mol),
        "murcko_scaffold": get_murcko_scaffold(smiles),
    })


def summarize_series(s: pd.Series, name: str):
    """Resumen corto y estable para comentar resultados numéricos."""
    numeric = pd.to_numeric(s, errors="coerce")
    summary = numeric.describe(percentiles=[0.25, 0.5, 0.75]).round(3)
    print(f"\n{name}")
    print(summary)

print("Dependencias químicas cargadas correctamente.")


Dependencias químicas cargadas correctamente.


## 3. Importación de moléculas generadas

Se parte del archivo `generated_molecules.csv`. En esta etapa solo se conservan moléculas válidas y no presentes exactamente en el conjunto de entrenamiento, si esas columnas existen.

Esta decisión evita dedicar recursos a moléculas inválidas o a copias exactas del dataset usado para entrenar/generar.


In [4]:
if not GENERATED_PATH.exists():
    raise FileNotFoundError(f"No existe el archivo de moléculas generadas: {GENERATED_PATH}")

df_generated_raw = pd.read_csv(GENERATED_PATH)
if "smiles" not in df_generated_raw.columns:
    raise ValueError("El archivo generated_molecules.csv debe contener una columna 'smiles'.")

# Se canonizan los SMILES para evitar duplicados escritos de forma distinta.
df_generated = df_generated_raw.copy()
df_generated["smiles"] = df_generated["smiles"].map(canonicalize_smiles)
df_generated = df_generated.dropna(subset=["smiles"]).drop_duplicates(subset="smiles").reset_index(drop=True)

# Si existen columnas del notebook generativo, se usan como filtros adicionales.
if "valid_rdkit" in df_generated.columns:
    df_generated = df_generated[df_generated["valid_rdkit"].astype(bool)].copy()
if "in_training_set" in df_generated.columns:
    df_generated = df_generated[~df_generated["in_training_set"].astype(bool)].copy()

df_generated = df_generated[["smiles"]].drop_duplicates().reset_index(drop=True)

print("Moléculas generadas válidas y no duplicadas:", len(df_generated))
df_generated.head()


Moléculas generadas válidas y no duplicadas: 9068


,smiles
0,Cc1ccc(C(=O)N2CC3CC2C3NC(=O)Cc2ccoc2)s1
1,Cc1cc(Br)c2c(c1)CN(C(=O)C1CCc3nc[nH]c3C1)CC2
2,Cc1ccc(-c2nc(CC(=O)N3CCN(C(=O)COc4ccc(Cl)cc4)C...
3,CC(=O)Nc1ccc2ncnc(Nc3cnn(CCN4CCOCC4)c3)c2c1
4,C=CCCC(C)N1CCC2(CC1)CCN(C1COC1)C2


## 4. Descriptores fisicoquímicos y filtro químico básico

En esta etapa se aplicó un primer filtro químico sobre las moléculas generadas con el objetivo de descartar estructuras poco plausibles como candidatas farmacológicas antes de realizar análisis más costosos, como la predicción de actividad, la evaluación ADMET y la selección por frente de Pareto.

Los criterios utilizados se basan en propiedades fisicoquímicas habituales en química médica: peso molecular, lipofilia, polaridad, número de donadores y aceptores de enlace de hidrógeno, enlaces rotables, número de átomos pesados, violaciones de la regla de Lipinski y QED. Estos filtros no pretenden demostrar que una molécula sea válida como fármaco, sino eliminar candidatos claramente desfavorables o poco realistas.

Se emplearon rangos relativamente permisivos para no descartar de forma prematura moléculas potencialmente interesantes. Por ejemplo, se permitió hasta dos violaciones de la regla de Lipinski, ya que algunos inhibidores de EGFR pueden presentar estructuras relativamente grandes o lipofílicas. Del mismo modo, el umbral mínimo de QED se fijó en 0.25 para eliminar moléculas con baja calidad farmacológica global sin imponer una selección excesivamente restrictiva en esta fase temprana.

Cada molécula fue evaluada individualmente frente a todos los criterios. Solo se conservaron aquellas que superaron todos los filtros básicos, generando el conjunto `df_filtered`. Este conjunto representa las moléculas químicamente más razonables para continuar con las siguientes etapas del pipeline de priorización.

Además, se calculó el número de filtros fallidos por molécula y el porcentaje de moléculas conservadas, lo que permite valorar el impacto del filtrado y detectar qué propiedades fueron más restrictivas en el conjunto generado.

In [5]:
desc = df_generated["smiles"].progress_apply(calc_basic_descriptors) if hasattr(pd.Series, "progress_apply") else df_generated["smiles"].apply(calc_basic_descriptors)
df_generated = pd.concat([df_generated.reset_index(drop=True), desc.reset_index(drop=True)], axis=1)

# Si la canonización del descriptor produjo una columna adicional, se mantiene 'smiles' como identificador principal.
df_generated.to_csv(DESCRIPTORS_OUTPUT_PATH, index=False)

summarize_series(df_generated["mol_wt"], "Peso molecular")
summarize_series(df_generated["rdkit_logp"], "RDKit LogP")
summarize_series(df_generated["tpsa"], "TPSA")
summarize_series(df_generated["qed"], "QED")
print("Scaffolds únicos:", df_generated["murcko_scaffold"].nunique())



Peso molecular
count    9068.000
mean      398.092
std        94.499
min       145.161
25%       335.407
50%       382.461
75%       445.450
max       991.093
Name: mol_wt, dtype: float64

RDKit LogP
count    9068.000
mean        3.350
std         1.665
min        -1.888
25%         2.257
50%         3.326
75%         4.410
max        11.131
Name: rdkit_logp, dtype: float64

TPSA
count    9068.000
mean       78.095
std        29.789
min         3.240
25%        58.120
50%        75.260
75%        94.840
max       281.510
Name: tpsa, dtype: float64

QED
count    9068.000
mean        0.622
std         0.207
min         0.026
25%         0.483
50%         0.664
75%         0.787
max         0.948
Name: qed, dtype: float64
Scaffolds únicos: 8265


In [6]:
BASIC_FILTERS = {
    "mol_wt_min": 180,
    "mol_wt_max": 650,
    "rdkit_logp_min": -1,
    "rdkit_logp_max": 6.5,
    "tpsa_min": 20,
    "tpsa_max": 160,
    "hbd_max": 5,
    "hba_max": 12,
    "rot_bonds_max": 12,
    "heavy_atoms_min": 10,
    "heavy_atoms_max": 70,
    "ro5_violations_max": 2,
    "qed_min": 0.25,
}

df_generated["pass_mw_filter"] = df_generated["mol_wt"].between(BASIC_FILTERS["mol_wt_min"], BASIC_FILTERS["mol_wt_max"])
df_generated["pass_logp_filter"] = df_generated["rdkit_logp"].between(BASIC_FILTERS["rdkit_logp_min"], BASIC_FILTERS["rdkit_logp_max"])
df_generated["pass_tpsa_filter"] = df_generated["tpsa"].between(BASIC_FILTERS["tpsa_min"], BASIC_FILTERS["tpsa_max"])
df_generated["pass_hbd_filter"] = df_generated["hbd"] <= BASIC_FILTERS["hbd_max"]
df_generated["pass_hba_filter"] = df_generated["hba"] <= BASIC_FILTERS["hba_max"]
df_generated["pass_rot_bonds_filter"] = df_generated["rot_bonds"] <= BASIC_FILTERS["rot_bonds_max"]
df_generated["pass_heavy_atoms_filter"] = df_generated["heavy_atoms"].between(BASIC_FILTERS["heavy_atoms_min"], BASIC_FILTERS["heavy_atoms_max"])
df_generated["pass_ro5_filter"] = df_generated["ro5_violations"] <= BASIC_FILTERS["ro5_violations_max"]
df_generated["pass_qed_filter"] = df_generated["qed"] >= BASIC_FILTERS["qed_min"]

basic_filter_cols = [
    "pass_mw_filter", "pass_logp_filter", "pass_tpsa_filter",
    "pass_hbd_filter", "pass_hba_filter", "pass_rot_bonds_filter",
    "pass_heavy_atoms_filter", "pass_ro5_filter", "pass_qed_filter",
]

df_generated["n_basic_filter_failures"] = (~df_generated[basic_filter_cols]).sum(axis=1)
df_generated["pass_basic_chemical_filter"] = df_generated["n_basic_filter_failures"].eq(0)

df_filtered = df_generated[df_generated["pass_basic_chemical_filter"]].copy().reset_index(drop=True)

df_filtered.to_csv(FILTERED_OUTPUT_PATH, index=False)

print("Moléculas iniciales:", len(df_generated))
print("Moléculas tras filtro químico básico:", len(df_filtered))
print("Porcentaje conservado:", round(100 * len(df_filtered) / max(len(df_generated), 1), 2), "%")

filter_summary = df_generated[basic_filter_cols].mean().sort_values().rename("fraction_pass").to_frame()
filter_summary


Moléculas iniciales: 9068
Moléculas tras filtro químico básico: 8241
Porcentaje conservado: 90.88 %


,fraction_pass
pass_qed_filter,0.933392
pass_logp_filter,0.964711
pass_mw_filter,0.978716
pass_tpsa_filter,0.979709
pass_rot_bonds_filter,0.991729
pass_ro5_filter,0.993052
pass_hbd_filter,0.997243
pass_hba_filter,0.997574
pass_heavy_atoms_filter,0.999890


En conjunto, estos resultados indican que el proceso generativo no produjo únicamente moléculas sintácticamente válidas, sino también estructuras con propiedades fisicoquímicas plausibles para continuar hacia etapas más exigentes de evaluación farmacológica.

In [7]:
print("Dataset filtrado guardado en:", FILTERED_OUTPUT_PATH)

Dataset filtrado guardado en: /home/kluna/TFM/egfr_pipeline_tfm/outputs/scoring/intermediate/02_generated_molecules_filtered.csv


## 5. Modelo QSAR para predicción de pIC50

Se entrena un modelo Random Forest sobre fingerprints Morgan. Es una elección razonable para un baseline robusto: no requiere redes neuronales adicionales, maneja relaciones no lineales.

La evaluación se realiza con split por scaffold de Murcko. Esto es más exigente que un split aleatorio porque reduce el riesgo de evaluar con moléculas casi idénticas a las de entrenamiento.


In [8]:
N_BITS = 2048
RADIUS = 2
MORGAN_GENERATOR = rdFingerprintGenerator.GetMorganGenerator(radius=RADIUS, fpSize=N_BITS)


def mol_to_morgan_array(smiles: str):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    arr = MORGAN_GENERATOR.GetFingerprintAsNumPy(mol)
    return arr.astype(np.int8)


def smiles_to_fp(smiles: str):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return MORGAN_GENERATOR.GetFingerprint(mol)


def scaffold_split_indices(df: pd.DataFrame, scaffold_col: str = "murcko_scaffold", frac_train=0.8, frac_valid=0.1, seed=42):
    rng = np.random.default_rng(seed)
    scaffold_groups = df.groupby(scaffold_col, dropna=False).indices
    scaffolds = list(scaffold_groups.keys())
    rng.shuffle(scaffolds)

    n_total = len(df)
    train_idx, valid_idx, test_idx = [], [], []

    for scaffold in scaffolds:
        group_idx = list(scaffold_groups[scaffold])
        if len(train_idx) / n_total < frac_train:
            train_idx.extend(group_idx)
        elif len(valid_idx) / n_total < frac_valid:
            valid_idx.extend(group_idx)
        else:
            test_idx.extend(group_idx)

    return np.array(train_idx), np.array(valid_idx), np.array(test_idx)


def regression_metrics(y_true, y_pred):
    metrics = {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }
    if spearmanr is not None:
        metrics["Spearman"] = float(spearmanr(y_true, y_pred).correlation)
    return metrics


In [9]:
if not REFERENCE_DATA_PATH.exists():
    raise FileNotFoundError(f"No existe el dataset de referencia para pIC50: {REFERENCE_DATA_PATH}")

df_ref = pd.read_csv(REFERENCE_DATA_PATH)
required_ref_cols = {"Smiles", "pIC50"}
missing_ref_cols = required_ref_cols - set(df_ref.columns)
if missing_ref_cols:
    raise ValueError(f"Faltan columnas en el dataset de referencia: {missing_ref_cols}")

df_ref = df_ref[["Smiles", "pIC50"]].rename(columns={"Smiles": "smiles"}).copy()
df_ref["smiles"] = df_ref["smiles"].map(canonicalize_smiles)
df_ref["pIC50"] = pd.to_numeric(df_ref["pIC50"], errors="coerce")
df_ref = df_ref.dropna(subset=["smiles", "pIC50"]).drop_duplicates(subset="smiles").reset_index(drop=True)

desc_ref = df_ref["smiles"].apply(calc_basic_descriptors)
df_ref = pd.concat([df_ref.reset_index(drop=True), desc_ref.reset_index(drop=True)], axis=1)

train_idx, valid_idx, test_idx = scaffold_split_indices(df_ref, scaffold_col="murcko_scaffold", seed=SEED)
df_ref["activity_split"] = "test"
df_ref.loc[train_idx, "activity_split"] = "train"
df_ref.loc[valid_idx, "activity_split"] = "valid"

print("Moléculas de referencia:", len(df_ref))
print(df_ref["activity_split"].value_counts())
print("Scaffolds por split:")
print(df_ref.groupby("activity_split")["murcko_scaffold"].nunique())


Moléculas de referencia: 10466
activity_split
train    8402
valid    1065
test      999
Name: count, dtype: int64
Scaffolds por split:
activity_split
test      430
train    3019
valid     392
Name: murcko_scaffold, dtype: int64


In [10]:
print("Calculando fingerprints Morgan para dataset de referencia...")
X_ref = np.vstack(df_ref["smiles"].map(mol_to_morgan_array).values)
y_ref = df_ref["pIC50"].astype(float).values

train_mask = df_ref["activity_split"].eq("train").values
valid_mask = df_ref["activity_split"].eq("valid").values
test_mask = df_ref["activity_split"].eq("test").values

X_train, y_train = X_ref[train_mask], y_ref[train_mask]
X_valid, y_valid = X_ref[valid_mask], y_ref[valid_mask]
X_test, y_test = X_ref[test_mask], y_ref[test_mask]

activity_model = RandomForestRegressor(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=SEED,
)
activity_model.fit(X_train, y_train)

valid_pred = activity_model.predict(X_valid)
test_pred = activity_model.predict(X_test)

activity_metrics = {
    "model": "RandomForestRegressor",
    "representation": f"Morgan radius={RADIUS}, n_bits={N_BITS}",
    "split": "Murcko scaffold split",
    "n_train": int(len(y_train)),
    "n_valid": int(len(y_valid)),
    "n_test": int(len(y_test)),
    "valid": regression_metrics(y_valid, valid_pred),
    "test": regression_metrics(y_test, test_pred),
    "created_at": datetime.now().isoformat(),
}

joblib.dump({"model": activity_model, "radius": RADIUS, "n_bits": N_BITS, "metrics": activity_metrics}, ACTIVITY_MODEL_PATH)
with open(ACTIVITY_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(activity_metrics, f, indent=4, ensure_ascii=False)

print("Modelo guardado en:", ACTIVITY_MODEL_PATH)
print("Métricas guardadas en:", ACTIVITY_METRICS_PATH)
activity_metrics


Calculando fingerprints Morgan para dataset de referencia...
Modelo guardado en: /home/kluna/TFM/egfr_pipeline_tfm/outputs/scoring/models/egfr_pic50_random_forest.pkl
Métricas guardadas en: /home/kluna/TFM/egfr_pipeline_tfm/outputs/scoring/models/egfr_pic50_random_forest_metrics.json


{'model': 'RandomForestRegressor',
 'representation': 'Morgan radius=2, n_bits=2048',
 'split': 'Murcko scaffold split',
 'n_train': 8402,
 'n_valid': 1065,
 'n_test': 999,
 'valid': {'MAE': 0.6483040398979637,
  'RMSE': 0.829728357252856,
  'R2': 0.5837658754931009,
  'Spearman': 0.772901327103646},
 'test': {'MAE': 0.6493200895659205,
  'RMSE': 0.8333415620777154,
  'R2': 0.6049353646087745,
  'Spearman': 0.7979729794937087},
 'created_at': '2026-05-12T15:39:11.763425'}

### Interpretación del modelo

El modelo se utilizará como herramienta de **ranking relativo**, no como estimador absoluto perfecto. Diferencias pequeñas de pIC50 no deben sobreinterpretarse. Por eso, más adelante el ranking combina actividad predicha con ADMET, similitud, QED y diversidad estructural.


## 6. Predicción de pIC50 en moléculas generadas

La predicción se aplica únicamente a moléculas que superaron el filtro químico básico. Así se evita priorizar moléculas que ya fallan por propiedades fisicoquímicas elementales.


In [11]:
if len(df_filtered) == 0:
    raise ValueError("No hay moléculas tras el filtro básico. Revisa los límites o la generación previa.")

print("Calculando fingerprints Morgan para moléculas filtradas...")
X_generated = np.vstack(df_filtered["smiles"].map(mol_to_morgan_array).values)

df_filtered["predicted_pIC50"] = activity_model.predict(X_generated)
df_filtered["predicted_active"] = df_filtered["predicted_pIC50"] >= 6.0
df_filtered["predicted_strong_active"] = df_filtered["predicted_pIC50"] >= 7.0

summarize_series(df_filtered["predicted_pIC50"], "pIC50 predicha")
print("Predichas activas pIC50 >= 6:", int(df_filtered["predicted_active"].sum()))
print("Predichas fuertemente activas pIC50 >= 7:", int(df_filtered["predicted_strong_active"].sum()))

df_filtered.head()


Calculando fingerprints Morgan para moléculas filtradas...

pIC50 predicha
count    8241.000
mean        6.140
std         0.316
min         5.215
25%         5.936
50%         6.108
75%         6.288
max         8.256
Name: predicted_pIC50, dtype: float64
Predichas activas pIC50 >= 6: 5445
Predichas fuertemente activas pIC50 >= 7: 162


,smiles,canonical_smiles,mol_wt,rdkit_logp,hbd,hba,tpsa,rot_bonds,rings,heavy_atoms,...,pass_hba_filter,pass_rot_bonds_filter,pass_heavy_atoms_filter,pass_ro5_filter,pass_qed_filter,n_basic_filter_failures,pass_basic_chemical_filter,predicted_pIC50,predicted_active,predicted_strong_active
0,Cc1ccc(C(=O)N2CC3CC2C3NC(=O)Cc2ccoc2)s1,Cc1ccc(C(=O)N2CC3CC2C3NC(=O)Cc2ccoc2)s1,330.409,2.22132,1,4,62.55,4,5,23,...,True,True,True,True,True,0,True,5.749222,False,False
1,Cc1cc(Br)c2c(c1)CN(C(=O)C1CCc3nc[nH]c3C1)CC2,Cc1cc(Br)c2c(c1)CN(C(=O)C1CCc3nc[nH]c3C1)CC2,374.282,3.17042,1,2,48.99,1,4,23,...,True,True,True,True,True,0,True,6.210604,True,False
2,Cc1ccc(-c2nc(CC(=O)N3CCN(C(=O)COc4ccc(Cl)cc4)C...,Cc1ccc(-c2nc(CC(=O)N3CCN(C(=O)COc4ccc(Cl)cc4)C...,469.994,4.06422,0,5,62.74,6,4,32,...,True,True,True,True,True,0,True,6.104071,True,False
3,CC(=O)Nc1ccc2ncnc(Nc3cnn(CCN4CCOCC4)c3)c2c1,CC(=O)Nc1ccc2ncnc(Nc3cnn(CCN4CCOCC4)c3)c2c1,381.440,1.86050,2,7,97.20,6,4,28,...,True,True,True,True,True,0,True,6.875740,True,False
4,N#Cc1cccc(OC2CCN(C3CCCC(OCc4ccccc4)C3)CC2)c1,N#Cc1cccc(OC2CCN(C3CCCC(OCc4ccccc4)C3)CC2)c1,390.527,4.92948,0,4,45.49,6,4,29,...,True,True,True,True,True,0,True,6.031031,True,False


Solo 162 moléculas alcanzan una pIC50 predicha igual o superior a 7. Esto es importante: el modelo genera muchas moléculas “razonablemente activas”, pero pocas con actividad alta. Por tanto, no genera candidatos excelentes, sino nos sirve como una primera priorización útil para identificar un subconjunto reducido de moléculas prometedoras.

## 7. Similitud frente al conjunto de entrenamiento

No basta con eliminar coincidencias exactas. Una molécula puede no estar en el training set y aun así ser prácticamente un análogo directo.

Se calcula la similitud Tanimoto máxima contra el training set para estimar:

- **novedad estructural**,
- **riesgo de copia**,
- **dominio de aplicabilidad del modelo QSAR**.

Una similitud muy baja puede indicar novedad, pero también que el modelo está extrapolando demasiado. Una similitud muy alta puede indicar copia o escasa innovación.


In [12]:
def nearest_tanimoto(query_smiles: pd.Series, reference_smiles: pd.Series, desc: str = "Tanimoto") -> pd.DataFrame:
    ref_df = pd.DataFrame({"ref_smiles": reference_smiles}).dropna().copy()
    ref_df["ref_smiles"] = ref_df["ref_smiles"].astype(str).map(canonicalize_smiles)
    ref_df = ref_df.dropna().drop_duplicates(subset="ref_smiles").reset_index(drop=True)
    ref_df["fp"] = ref_df["ref_smiles"].map(smiles_to_fp)
    ref_df = ref_df[ref_df["fp"].notna()].reset_index(drop=True)

    ref_fps = ref_df["fp"].tolist()
    ref_smiles_list = ref_df["ref_smiles"].tolist()

    rows = []
    for smi in tqdm(query_smiles.astype(str), desc=desc):
        fp = smiles_to_fp(smi)
        if fp is None or not ref_fps:
            rows.append({"max_tanimoto": np.nan, "nearest_smiles": None})
            continue
        sims = DataStructs.BulkTanimotoSimilarity(fp, ref_fps)
        best_idx = int(np.argmax(sims))
        rows.append({
            "max_tanimoto": float(sims[best_idx]),
            "nearest_smiles": ref_smiles_list[best_idx],
        })
    return pd.DataFrame(rows)



In [13]:

if not TRAINING_DATA_PATH.exists():
    raise FileNotFoundError(f"No existe el dataset de entrenamiento/generación: {TRAINING_DATA_PATH}")

df_train = pd.read_csv(TRAINING_DATA_PATH)
smiles_col = "smiles" if "smiles" in df_train.columns else "Smiles"
if smiles_col not in df_train.columns:
    raise ValueError("El dataset de entrenamiento debe contener una columna 'smiles' o 'Smiles'.")

df_train = df_train[[smiles_col]].rename(columns={smiles_col: "smiles"}).copy()
df_train["smiles"] = df_train["smiles"].map(canonicalize_smiles)
df_train = df_train.dropna(subset=["smiles"]).drop_duplicates(subset="smiles").reset_index(drop=True)

train_sim = nearest_tanimoto(df_filtered["smiles"], df_train["smiles"], desc="Similitud vs training set")
df_filtered["max_tanimoto_training"] = train_sim["max_tanimoto"].values
df_filtered["nearest_training_smiles"] = train_sim["nearest_smiles"].values

summarize_series(df_filtered["max_tanimoto_training"], "Máxima similitud Tanimoto vs training")


Similitud vs training set:   0%|          | 0/8241 [00:00<?, ?it/s]


Máxima similitud Tanimoto vs training
count    8241.000
mean        0.300
std         0.087
min         0.114
25%         0.247
50%         0.281
75%         0.326
max         1.000
Name: max_tanimoto_training, dtype: float64


Los resultados muestran una similitud media de 0.30 y una mediana de 0.28, indicando que la mayoría de las moléculas presentan una similitud moderada o baja respecto a las moléculas empleadas durante el entrenamiento. Además, el 75 % de las moléculas tienen una similitud inferior a 0.33, lo que sugiere que el modelo exploró regiones relativamente nuevas del espacio químico.

In [14]:
bins = [0, 0.25, 0.35, 0.50, 0.70, 0.85, 0.95, 1.01]
labels = [
    "<0.25 muy lejanas",
    "0.25-0.35 lejanas",
    "0.35-0.50 novedosas razonables",
    "0.50-0.70 moderadamente similares",
    "0.70-0.85 análogos cercanos",
    "0.85-0.95 casi copias",
    ">0.95 copias/casi copias",
]

df_filtered["training_similarity_bin"] = pd.cut(
    df_filtered["max_tanimoto_training"],
    bins=bins,
    labels=labels,
    include_lowest=True,
)

similarity_activity_summary = df_filtered.groupby("training_similarity_bin", observed=False)["predicted_pIC50"].agg(
    ["count", "mean", "median", "min", "max"]
).round(3)

similarity_activity_summary


,count,mean,median,min,max
training_similarity_bin,,,,,
<0.25 muy lejanas,2331,6.007,6.012,5.391,6.723
0.25-0.35 lejanas,4463,6.110,6.111,5.215,6.908
0.35-0.50 novedosas razonables,1144,6.314,6.298,5.366,7.574
0.50-0.70 moderadamente similares,256,6.910,6.887,5.797,8.089
0.70-0.85 análogos cercanos,40,7.066,7.083,6.578,8.174
0.85-0.95 casi copias,5,7.754,7.488,7.425,8.256
>0.95 copias/casi copias,2,7.474,7.474,6.966,7.982


Se observa una relación clara entre similitud estructural y actividad predicha: las moléculas más parecidas al conjunto de entrenamiento tienden a presentar mayores valores de pIC50. Las estructuras muy novedosas muestran, en promedio, menor actividad estimada, mientras que los compuestos moderadamente similares (`0.35–0.70`) mantienen un equilibrio más interesante entre novedad química y potencial bioactivo.

Estos resultados sugieren que el modelo no solo reprodujo moléculas conocidas, sino que también fue capaz de generar estructuras relativamente nuevas conservando niveles razonables de actividad frente a EGFR.

## 8. Predicciones ADMET-AI

La predicción de propiedades ADMET se realizó utilizando la herramienta abierta **ADMET-AI**, desarrollada por el grupo de investigación de Swanson Lab. Esta herramienta permite estimar múltiples propiedades farmacocinéticas y toxicológicas a partir de la estructura molecular, facilitando una evaluación temprana de la calidad farmacológica de los compuestos generados.

Repositorio oficial del proyecto:  
[ADMET-AI GitHub Repository](https://github.com/swansonk14/admet_ai?utm_source=chatgpt.com)


In [15]:
RUN_ADMET_AI = True          # Cambiar a False si se desea reutilizar un CSV existente o saltar esta etapa.
ADMET_BATCH_SIZE = 1000

admet_input = df_filtered[["smiles"]].drop_duplicates().reset_index(drop=True)
admet_input.to_csv(ADMET_INPUT_PATH, index=False)

print("Moléculas únicas para ADMET-AI:", len(admet_input))
print("Input ADMET guardado en:", ADMET_INPUT_PATH)


Moléculas únicas para ADMET-AI: 8241
Input ADMET guardado en: /home/kluna/TFM/egfr_pipeline_tfm/outputs/scoring/intermediate/03_admet_ai_input.csv


In [16]:
def predict_admet_if_available(admet_input: pd.DataFrame) -> pd.DataFrame:
    """Ejecuta ADMET-AI si está disponible. Si no, intenta leer predicciones previas."""
    if ADMET_OUTPUT_PATH.exists() and not RUN_ADMET_AI:
        print("RUN_ADMET_AI=False. Leyendo predicciones existentes:", ADMET_OUTPUT_PATH)
        return pd.read_csv(ADMET_OUTPUT_PATH)

    if not RUN_ADMET_AI:
        warnings.warn("ADMET-AI desactivado y no se ejecutarán predicciones nuevas.")
        return admet_input.copy()

    try:
        from admet_ai import ADMETModel
        from admet_ai.constants import DEFAULT_MODELS_DIR, DEFAULT_DRUGBANK_PATH
    except ImportError:
        if ADMET_OUTPUT_PATH.exists():
            warnings.warn("ADMET-AI no está instalado. Se reutilizan predicciones existentes.")
            return pd.read_csv(ADMET_OUTPUT_PATH)
        warnings.warn("ADMET-AI no está instalado y no existe archivo previo. Se continúa sin columnas ADMET.")
        return admet_input.copy()

    print("Cargando modelo ADMET-AI...")
    model = ADMETModel(
        models_dir=DEFAULT_MODELS_DIR,
        include_physchem=True,
        drugbank_path=DEFAULT_DRUGBANK_PATH,
        atc_code=None,
        num_workers=0,
    )

    all_preds = []
    smiles_list = admet_input["smiles"].astype(str).tolist()
    for start in tqdm(range(0, len(smiles_list), ADMET_BATCH_SIZE), desc="ADMET-AI batches"):
        batch_smiles = smiles_list[start:start + ADMET_BATCH_SIZE]
        preds = model.predict(smiles=batch_smiles)
        batch_df = pd.DataFrame(preds) if not isinstance(preds, pd.DataFrame) else preds.copy()
        if "smiles" not in batch_df.columns:
            batch_df.insert(0, "smiles", batch_smiles[:len(batch_df)])
        all_preds.append(batch_df)

    df_admet = pd.concat(all_preds, ignore_index=True)
    df_admet["smiles"] = df_admet["smiles"].astype(str).map(canonicalize_smiles)
    df_admet = df_admet.dropna(subset=["smiles"]).drop_duplicates(subset="smiles").reset_index(drop=True)
    df_admet.to_csv(ADMET_OUTPUT_PATH, index=False)
    print("Predicciones ADMET guardadas en:", ADMET_OUTPUT_PATH)
    return df_admet


In [17]:
df_admet = predict_admet_if_available(admet_input)

df_filtered = df_filtered.merge(df_admet, on="smiles", how="left", suffixes=("", "_admet"))
print("Shape tras merge ADMET:", df_filtered.shape)

Cargando modelo ADMET-AI...


ADMET-AI batches:   0%|          | 0/9 [00:00<?, ?it/s]

Computing physchem properties: 100%|██████████| 1000/1000 [00:03<00:00, 251.63it/s]
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

/home/kluna/TFM/egfr_pipeline_tfm/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/kluna/TFM/egfr_pipeline_tfm/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


...

Output()

Output()

Output()

model ensembles: 100%|██████████| 2/2 [00:02<00:00,  1.32s/it]


Predicciones ADMET guardadas en: /home/kluna/TFM/egfr_pipeline_tfm/outputs/scoring/intermediate/04_admet_ai_predictions.csv
Shape tras merge ADMET: (8241, 134)


## 9. Score ADMET y filtros duros

ADMET-AI puede devolver muchas columnas. Para evitar doble conteo, se descartan columnas fisicoquímicas duplicadas que ya fueron calculadas con RDKit. Después se construyen dos señales distintas:

- `admet_score`: score blando agregado, usado para ranking.
- `admet_hard_pass_fraction`: fracción de filtros críticos superados, usada para controlar riesgos fuertes.

Separar ambas señales evita el error de esconder una alerta grave dentro de una media global aparentemente aceptable.


In [18]:
admet_duplicate_cols = [
    "molecular_weight", "logP", "hydrogen_bond_acceptors", "hydrogen_bond_donors",
    "Lipinski", "QED", "tpsa_admet",
    "molecular_weight_drugbank_approved_percentile",
    "logP_drugbank_approved_percentile",
    "hydrogen_bond_acceptors_drugbank_approved_percentile",
    "hydrogen_bond_donors_drugbank_approved_percentile",
    "Lipinski_drugbank_approved_percentile",
    "QED_drugbank_approved_percentile",
    "tpsa_drugbank_approved_percentile",
]
cols_to_drop = [c for c in admet_duplicate_cols if c in df_filtered.columns]
df_scored = df_filtered.drop(columns=cols_to_drop).copy()


Se aplicaron criterios ADMET estrictos con el objetivo de eliminar moléculas con señales tempranas de toxicidad, baja biodisponibilidad o problemas químicos conocidos. En particular, se descartaron compuestos con alertas estructurales relevantes (`PAINS, NIH y BRENK`), así como moléculas con riesgo elevado de mutagenicidad, cardiotoxicidad o carcinogenicidad según las predicciones de ADMET-AI.

Además, se exigieron valores mínimos de biodisponibilidad oral y absorción intestinal humana (`Bioavailability_Ma` y `HIA_Hou`) para priorizar candidatos con mejores perspectivas farmacocinéticas.

In [19]:

HARD_ADMET_RULES = {
    "PAINS_alert": {"type": "max", "threshold": 0},
    "NIH_alert": {"type": "max", "threshold": 0},
    "BRENK_alert": {"type": "max", "threshold": 1},
    "AMES": {"type": "max", "threshold": 0.5},
    "ClinTox": {"type": "max", "threshold": 0.5},
    "Carcinogens_Lagunin": {"type": "max", "threshold": 0.5},
    "hERG": {"type": "max", "threshold": 0.5},
    "Bioavailability_Ma": {"type": "min", "threshold": 0.5},
    "HIA_Hou": {"type": "min", "threshold": 0.5},
}


def apply_hard_admet_filters(df: pd.DataFrame, rules: dict) -> pd.DataFrame:
    df = df.copy()
    pass_cols = []
    for col, rule in rules.items():
        if col not in df.columns:
            warnings.warn(f"Columna ADMET no encontrada: {col}")
            continue
        pass_col = f"pass_{col}"
        values = pd.to_numeric(df[col], errors="coerce")
        if rule["type"] == "max":
            df[pass_col] = values <= rule["threshold"]
        elif rule["type"] == "min":
            df[pass_col] = values >= rule["threshold"]
        else:
            raise ValueError(f"Tipo de regla desconocido para {col}: {rule['type']}")
        pass_cols.append(pass_col)

    if pass_cols:
        df["admet_hard_pass_count"] = df[pass_cols].sum(axis=1)
        df["admet_hard_pass_fraction"] = df["admet_hard_pass_count"] / len(pass_cols)
        df["admet_hard_pass_all"] = df[pass_cols].all(axis=1)
    else:
        df["admet_hard_pass_count"] = np.nan
        df["admet_hard_pass_fraction"] = 0.0
        df["admet_hard_pass_all"] = False
    return df

df_scored = apply_hard_admet_filters(df_scored, HARD_ADMET_RULES)
summarize_series(df_scored["admet_hard_pass_fraction"], "Fracción de filtros ADMET duros superados")



Fracción de filtros ADMET duros superados
count    8241.000
mean        0.880
std         0.098
min         0.333
25%         0.778
50%         0.889
75%         1.000
max         1.000
Name: admet_hard_pass_fraction, dtype: float64


### 9.1. Filtros blandos

Para integrar las predicciones ADMET-AI se construyó un score agregado basado en percentiles frente a moléculas aprobadas de DrugBank. Para endpoints de riesgo, como hERG, AMES, ClinTox, carcinogenicidad, DILI y reacciones cutáneas, se consideraron preferibles percentiles bajos. Para propiedades relacionadas con absorción y exposición, como biodisponibilidad, absorción intestinal, permeabilidad y solubilidad, se consideraron preferibles percentiles altos. Cada componente fue transformado a una escala 0–1 y combinado mediante una media ponderada, asignando mayor peso a riesgos críticos como hERG y ClinTox, y menor peso a interacciones CYP, que se trataron como penalizaciones blandas

In [20]:
def percentile_score_component(df: pd.DataFrame, col: str, direction: str) -> pd.Series:
    s = pd.to_numeric(df[col], errors="coerce")
    if direction == "max":
        return s.clip(0, 1)
    if direction == "min":
        return (1 - s).clip(0, 1)
    raise ValueError(f"Dirección no reconocida: {direction}")


def build_weighted_admet_score(df: pd.DataFrame, objectives: dict) -> pd.Series:
    score = pd.Series(0.0, index=df.index)
    total_weight = 0.0
    for col, cfg in objectives.items():
        if col not in df.columns:
            warnings.warn(f"Columna ADMET para score no encontrada: {col}")
            continue
        weight = float(cfg.get("weight", 1.0))
        score += weight * percentile_score_component(df, col, cfg["direction"]).fillna(0.0)
        total_weight += weight
    if total_weight == 0:
        return pd.Series(0.0, index=df.index)
    return score / total_weight

# Objetivos blandos. Se usan percentiles cuando existen; si alguna columna falta, se ignora con warning.
ADMET_SCORE_OBJECTIVES = {
    # Toxicidad / seguridad: menor percentil es mejor
    "AMES_drugbank_approved_percentile": {"direction": "min", "weight": 1.0},
    "ClinTox_drugbank_approved_percentile": {"direction": "min", "weight": 1.2},
    "Carcinogens_Lagunin_drugbank_approved_percentile": {"direction": "min", "weight": 1.0},
    "hERG_drugbank_approved_percentile": {"direction": "min", "weight": 1.5},
    "DILI_drugbank_approved_percentile": {"direction": "min", "weight": 0.7},
    "Skin_Reaction_drugbank_approved_percentile": {"direction": "min", "weight": 0.5},

    # Absorción / exposición: mayor percentil es mejor
    "Bioavailability_Ma_drugbank_approved_percentile": {"direction": "max", "weight": 1.0},
    "HIA_Hou_drugbank_approved_percentile": {"direction": "max", "weight": 0.8},
    "PAMPA_NCATS_drugbank_approved_percentile": {"direction": "max", "weight": 0.5},
    "Caco2_Wang_drugbank_approved_percentile": {"direction": "max", "weight": 0.5},
    "Solubility_AqSolDB_drugbank_approved_percentile": {"direction": "max", "weight": 1.0},

    # Metabolismo CYP: menor inhibición suele ser mejor, pero peso bajo
    "CYP1A2_Veith_drugbank_approved_percentile": {"direction": "min", "weight": 0.3},
    "CYP2C19_Veith_drugbank_approved_percentile": {"direction": "min", "weight": 0.3},
    "CYP2C9_Veith_drugbank_approved_percentile": {"direction": "min", "weight": 0.3},
    "CYP2D6_Veith_drugbank_approved_percentile": {"direction": "min", "weight": 0.3},
    "CYP3A4_Veith_drugbank_approved_percentile": {"direction": "min", "weight": 0.3},
}

df_scored["admet_score"] = build_weighted_admet_score(df_scored, ADMET_SCORE_OBJECTIVES)
summarize_series(df_scored["admet_score"], "Score ADMET blando")



Score ADMET blando
count    8241.000
mean        0.339
std         0.004
min         0.263
25%         0.339
50%         0.339
75%         0.339
max         0.385
Name: admet_score, dtype: float64


## 10. Scores finales interpretables

Se construyen scores en escala 0–1 para combinar criterios heterogéneos. Esto evita comparar directamente variables con unidades distintas.

Decisiones clave:

- La actividad se transforma de forma lineal entre pIC50 6.0 y 7.5.
- La similitud al training se premia en una ventana intermedia: ni demasiado lejos ni demasiado cerca.
- QED se usa directamente porque ya está entre 0 y 1.
- RO5 se penaliza según número de violaciones.
- ADMET se separa en score blando y filtros duros.


In [21]:
def clipped_linear_score(s: pd.Series, low: float, high: float) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    return ((s - low) / (high - low)).clip(0, 1).fillna(0.0)


def similarity_window_score(s: pd.Series, low=0.35, high=0.85) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    score = pd.Series(0.0, index=s.index)
    in_window = (s >= low) & (s <= high)
    below = s < low
    above = s > high
    score[in_window] = 1.0
    score[below] = (s[below] / low).clip(0, 1)
    score[above] = (1.0 - ((s[above] - high) / (1.0 - high))).clip(0, 1)
    return score.fillna(0.0)

df = df_scored.copy()

df["is_exact_training_match"] = df["smiles"].eq(df["nearest_training_smiles"])
df["flag_near_copy_training"] = df["max_tanimoto_training"] >= 0.95
df["flag_too_far_training"] = df["max_tanimoto_training"] < 0.30
df["flag_good_training_window"] = df["max_tanimoto_training"].between(0.35, 0.85)

df["score_activity"] = clipped_linear_score(df["predicted_pIC50"], low=6.0, high=7.5)
df["score_training_similarity"] = similarity_window_score(df["max_tanimoto_training"], low=0.35, high=0.85)
df["score_qed"] = pd.to_numeric(df["qed"], errors="coerce").clip(0, 1).fillna(0.0)
df["score_ro5"] = (1.0 - (pd.to_numeric(df["ro5_violations"], errors="coerce") / 2.0)).clip(0, 1).fillna(0.0)
df["score_admet"] = pd.to_numeric(df["admet_score"], errors="coerce").clip(0, 1).fillna(0.0)
df["score_admet_hard"] = pd.to_numeric(df["admet_hard_pass_fraction"], errors="coerce").clip(0, 1).fillna(0.0)

print("Coincidencias exactas con training:", int(df["is_exact_training_match"].sum()))
print("Casi copias training >= 0.95:", int(df["flag_near_copy_training"].sum()))
print("Fuera de dominio aproximado < 0.30:", int(df["flag_too_far_training"].sum()))


Coincidencias exactas con training: 0
Casi copias training >= 0.95: 2
Fuera de dominio aproximado < 0.30: 5100



| Componente         | Peso | Motivo                                          |
| ------------------ | ---: | ----------------------------------------------- |
| Actividad          | 0.30 | El objetivo es inhibición EGFR                  |
| ADMET blando       | 0.20 | Evita elegir moléculas farmacológicamente malas |
| ADMET duro         | 0.10 | Penaliza riesgos críticos                       |
| Similitud training | 0.15 | Equilibra novedad y aplicabilidad               |
| QED                | 0.15 | Drug-likeness general                           |
| RO5                | 0.10 | Control fisicoquímico básico                    |


In [22]:
REWARD_WEIGHTS = {
    "score_activity": 0.30,
    "score_admet": 0.20,
    "score_admet_hard": 0.10,
    "score_training_similarity": 0.15,
    "score_qed": 0.15,
    "score_ro5": 0.10,
}

reward = pd.Series(0.0, index=df.index)
for col, weight in REWARD_WEIGHTS.items():
    reward += weight * pd.to_numeric(df[col], errors="coerce").fillna(0.0)
df["reward_score"] = reward / sum(REWARD_WEIGHTS.values())

summarize_series(df["reward_score"], "Reward score")

df[["smiles", "predicted_pIC50", "admet_score", "admet_hard_pass_fraction", "max_tanimoto_training", "qed", "ro5_violations", "reward_score"]].head()



Reward score
count    8241.000
mean        0.506
std         0.060
min         0.301
25%         0.472
50%         0.501
75%         0.535
max         0.814
Name: reward_score, dtype: float64


,smiles,predicted_pIC50,admet_score,admet_hard_pass_fraction,max_tanimoto_training,qed,ro5_violations,reward_score
0,Cc1ccc(C(=O)N2CC3CC2C3NC(=O)Cc2ccoc2)s1,5.749222,0.339286,1.000000,0.184466,0.934864,0,0.487144
1,Cc1cc(Br)c2c(c1)CN(C(=O)C1CCc3nc[nH]c3C1)CC2,6.210604,0.339286,0.888889,0.229730,0.833758,0,0.522386
2,Cc1ccc(-c2nc(CC(=O)N3CCN(C(=O)COc4ccc(Cl)cc4)C...,6.104071,0.339286,0.888889,0.321839,0.545740,0,0.497352
3,CC(=O)Nc1ccc2ncnc(Nc3cnn(CCN4CCOCC4)c3)c2c1,6.875740,0.339286,0.666667,0.569444,0.672545,0,0.660554
4,N#Cc1cccc(OC2CCN(C3CCCC(OCc4ccccc4)C3)CC2)c1,6.031031,0.339286,0.888889,0.276596,0.701556,0,0.486727


## 11. Pool elegible

Antes de aplicar Pareto se define un conjunto elegible. Esta etapa evita que el frente Pareto conserve moléculas que son formalmente no dominadas, pero estratégicamente débiles.

Criterios usados:

- pasan el filtro químico básico,
- no son coincidencias exactas ni casi copias del training,
- no están demasiado lejos del dominio del modelo,
- tienen actividad predicha razonable,
- superan una proporción suficiente de filtros ADMET duros.


In [23]:
ELIGIBILITY_THRESHOLDS = {
    "min_tanimoto_training": 0.30,
    "min_predicted_pIC50": 6.0,
    "min_admet_hard_pass_fraction": 0.75,
}

df_eligible = df[
    df["pass_basic_chemical_filter"].astype(bool)
    & ~df["is_exact_training_match"].astype(bool)
    & ~df["flag_near_copy_training"].astype(bool)
    & (df["max_tanimoto_training"] >= ELIGIBILITY_THRESHOLDS["min_tanimoto_training"])
    & (df["predicted_pIC50"] >= ELIGIBILITY_THRESHOLDS["min_predicted_pIC50"])
    & (df["admet_hard_pass_fraction"] >= ELIGIBILITY_THRESHOLDS["min_admet_hard_pass_fraction"])
].copy()

print("Moléculas tras scoring completo:", len(df))
print("Moléculas elegibles:", len(df_eligible))
print("Porcentaje elegible:", round(100 * len(df_eligible) / max(len(df), 1), 2), "%")


Moléculas tras scoring completo: 8241
Moléculas elegibles: 2133
Porcentaje elegible: 25.88 %


## 12. Selección Pareto y diversidad por scaffold

La selección Pareto evita elegir candidatos solo por un único score. Una molécula pertenece al frente Pareto si ninguna otra es mejor o igual en todos los objetivos y estrictamente mejor en al menos uno.

Después se aplica un límite por scaffold para evitar que el top final esté dominado por variantes de un mismo núcleo químico.


In [24]:
def pareto_front(df: pd.DataFrame, objectives: list[tuple[str, str]]) -> pd.Series:
    work = df[[col for col, _ in objectives]].copy()
    for col, direction in objectives:
        work[col] = pd.to_numeric(work[col], errors="coerce")
        if direction == "min":
            work[col] = -work[col]

    values = work.fillna(-np.inf).values
    n = values.shape[0]
    is_efficient = np.ones(n, dtype=bool)

    for i in range(n):
        if not is_efficient[i]:
            continue
        dominated_by_other = np.all(values >= values[i], axis=1) & np.any(values > values[i], axis=1)
        if np.any(dominated_by_other):
            is_efficient[i] = False

    return pd.Series(is_efficient, index=df.index)


In [25]:
pareto_objectives = [
    ("predicted_pIC50", "max"),
    ("score_admet", "max"),
    ("score_admet_hard", "max"),
    ("score_training_similarity", "max"),
    ("score_qed", "max"),
    ("score_ro5", "max"),
]

if len(df_eligible) == 0:
    warnings.warn("No hay moléculas elegibles. Se usará el dataframe completo para evitar salida vacía; revisa umbrales.")
    pareto_source = df.copy()
else:
    pareto_source = df_eligible.copy()

pareto_mask = pareto_front(pareto_source, pareto_objectives)
df_pareto = pareto_source[pareto_mask].copy()

print("Moléculas elegibles:", len(df_eligible))
print("Moléculas en frente Pareto:", len(df_pareto))

Moléculas elegibles: 2133
Moléculas en frente Pareto: 37


In [26]:
MAX_PER_SCAFFOLD = 3
TOP_N = 100
TOP_SHORTLIST = 30

df_top100 = (
    df_pareto
    .sort_values("reward_score", ascending=False)
    .groupby("murcko_scaffold", dropna=False)
    .head(MAX_PER_SCAFFOLD)
    .sort_values("reward_score", ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
)

df_shortlist = df_top100.head(TOP_SHORTLIST).copy().reset_index(drop=True)
df_shortlist.insert(0, "rank", np.arange(1, len(df_shortlist) + 1))

print("Moléculas en Pareto:", len(df_pareto))
print("Top diverso:", len(df_top100))
print("Shortlist final:", len(df_shortlist))

df_shortlist[[
        "smiles",
        "predicted_pIC50",
        "reward_score",
        "admet_score",
        "admet_hard_pass_fraction",
        "max_tanimoto_training",
        "qed",
        "ro5_violations",
        "murcko_scaffold",
        "AMES",
        "ClinTox",
        "hERG",
        "Bioavailability_Ma",
        "HIA_Hou",
    ]].head(30)


Moléculas en Pareto: 37
Top diverso: 37
Shortlist final: 30


,smiles,predicted_pIC50,reward_score,admet_score,admet_hard_pass_fraction,max_tanimoto_training,qed,ro5_violations,murcko_scaffold,AMES,ClinTox,hERG,Bioavailability_Ma,HIA_Hou
0,COc1ccc2ncnc(Nc3cccc(Cl)c3F)c2c1,7.678820,0.813901,0.339286,0.777778,0.603774,0.788442,0,c1ccc(Nc2ncnc3ccccc23)cc1,0.739540,0.181983,0.775938,0.915025,0.999996
1,O=C(Nc1ccc2ncnc(Nc3cccc(Br)c3)c2c1)C1CCOCC1,7.447770,0.792513,0.339286,0.888889,0.698413,0.641418,0,O=C(Nc1ccc2ncnc(Nc3ccccc3)c2c1)C1CCOCC1,0.385692,0.366434,0.779982,0.885770,0.999588
2,COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OC1CC2(COC2)C1,7.360564,0.776262,0.339286,0.888889,0.706897,0.649353,0,c1ccc(Nc2ncnc3ccc(OC4CC5(COC5)C4)cc23)cc1,0.432403,0.320576,0.855299,0.932424,0.999965
3,O=C(Nc1cc2c(Nc3cccc(Br)c3)ncnc2cn1)Nc1nc2n(n1)...,7.768092,0.767045,0.339286,0.888889,0.557143,0.401990,0,O=C(Nc1cc2c(Nc3ccccc3)ncnc2cn1)Nc1nc2n(n1)CCCC2,0.220596,0.359608,0.777587,0.889930,0.998413
4,C=CC(=O)Nc1cc(Nc2ncc(C)cn2)c(OC)cc1N1CCOCC1,7.182763,0.756761,0.339286,0.888889,0.613333,0.756415,0,c1cnc(Nc2ccc(N3CCOCC3)cc2)nc1,0.143175,0.350997,0.437345,0.916171,0.991351
5,CC(C)n1cnc2ncnc(Nc3cccc(CO)c3)c21,6.852217,0.703727,0.339286,1.000000,0.365079,0.769510,0,c1ccc(Nc2ncnc3nc[nH]c23)cc1,0.312633,0.239732,0.440849,0.945616,0.999707
6,CCOc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1NC(=O)/C=C/...,8.170641,0.701087,0.339286,0.777778,0.861111,0.443757,1,O=C(/C=C/CN1CCCC1)Nc1ccc2ncnc(Nc3ccccc3)c2c1,0.434214,0.701487,0.915025,0.719068,0.994625
7,COc1cc2ncnc(Nc3cc(F)ccc3F)c2cc1OC,6.930272,0.701011,0.339286,0.777778,0.733333,0.795479,0,c1ccc(Nc2ncnc3ccccc23)cc1,0.730740,0.185598,0.703799,0.962090,0.999994
8,CC(=O)NC1CCN(C(=O)C23CC(CN2c2ncnc4cc(Cl)ccc24)...,6.779645,0.690375,0.339286,1.000000,0.329670,0.835345,0,O=C(N1CCCCC1)C12CC(CN1c1ncnc3ccccc13)C2,0.218085,0.216504,0.404341,0.862336,0.999703
9,CC(=O)N1CCC(C(=O)Nc2cc(Nc3ccc(F)c(Cl)c3)ncn2)CC1,6.782786,0.688581,0.339286,0.888889,0.520548,0.835187,0,O=C(Nc1cc(Nc2ccccc2)ncn1)C1CCNCC1,0.325117,0.442769,0.614251,0.918820,0.998351


## 13. Comparación frente a fármacos EGFR de referencia

Si existe un archivo `fda_ema_egfr_reference.csv` con columnas `drug_name` y `smiles`, se calcula la similitud máxima de cada candidata frente a ese panel.

Esta comparación no se usa para demostrar eficacia, sino para identificar si el top 30 contiene análogos demasiado obvios de moléculas regulatoriamente conocidas. Si el archivo no existe, el notebook continúa sin esta etapa.


In [27]:
def nearest_tanimoto_with_labels(query_smiles: pd.Series, reference_smiles: pd.Series, reference_labels: pd.Series, desc: str = "Tanimoto") -> pd.DataFrame:
    ref_df = pd.DataFrame({"ref_smiles": reference_smiles, "ref_label": reference_labels}).dropna(subset=["ref_smiles"]).copy()
    ref_df["ref_smiles"] = ref_df["ref_smiles"].astype(str).map(canonicalize_smiles)
    ref_df["ref_label"] = ref_df["ref_label"].astype(str)
    ref_df = ref_df.dropna(subset=["ref_smiles"]).drop_duplicates(subset="ref_smiles").reset_index(drop=True)
    ref_df["fp"] = ref_df["ref_smiles"].map(smiles_to_fp)
    ref_df = ref_df[ref_df["fp"].notna()].reset_index(drop=True)

    ref_fps = ref_df["fp"].tolist()
    ref_smiles_list = ref_df["ref_smiles"].tolist()
    ref_label_list = ref_df["ref_label"].tolist()

    rows = []
    for smi in tqdm(query_smiles.astype(str), desc=desc):
        fp = smiles_to_fp(smi)
        if fp is None or not ref_fps:
            rows.append({"max_tanimoto_approved_egfr": np.nan, "nearest_approved_egfr_smiles": None, "nearest_approved_egfr_name": None})
            continue
        sims = DataStructs.BulkTanimotoSimilarity(fp, ref_fps)
        best_idx = int(np.argmax(sims))
        rows.append({
            "max_tanimoto_approved_egfr": float(sims[best_idx]),
            "nearest_approved_egfr_smiles": ref_smiles_list[best_idx],
            "nearest_approved_egfr_name": ref_label_list[best_idx],
        })
    return pd.DataFrame(rows)


def approved_similarity_category(x):
    if pd.isna(x):
        return "unknown"
    if x >= 0.90:
        return "casi clon de aprobado"
    if x >= 0.70:
        return "análogo cercano de aprobado"
    if x >= 0.40:
        return "relacionado medicinalmente"
    return "novedad alta frente a aprobados"

if APPROVED_EGFR_REF_PATH.exists() and len(df_shortlist) > 0:
    df_approved_egfr = pd.read_csv(APPROVED_EGFR_REF_PATH)
    required_cols = {"drug_name", "smiles"}
    missing = required_cols - set(df_approved_egfr.columns)
    if missing:
        warnings.warn(f"El archivo de referencia EGFR existe pero faltan columnas: {missing}. Se omite comparación.")
    else:
        approved_sim = nearest_tanimoto_with_labels(
            query_smiles=df_shortlist["smiles"],
            reference_smiles=df_approved_egfr["smiles"],
            reference_labels=df_approved_egfr["drug_name"],
            desc="Similitud vs EGFR aprobados/referencia",
        )
        df_shortlist = pd.concat([df_shortlist.reset_index(drop=True), approved_sim.reset_index(drop=True)], axis=1)
        df_shortlist["approved_egfr_similarity_category"] = df_shortlist["max_tanimoto_approved_egfr"].map(approved_similarity_category)
        print(df_shortlist["approved_egfr_similarity_category"].value_counts(dropna=False))
else:
    print("No se encontró referencia FDA/EMA EGFR. Se omite esta comparación opcional.")
    df_shortlist["max_tanimoto_approved_egfr"] = np.nan
    df_shortlist["nearest_approved_egfr_name"] = None
    df_shortlist["approved_egfr_similarity_category"] = "not_evaluated"


Similitud vs EGFR aprobados/referencia:   0%|          | 0/30 [00:00<?, ?it/s]

approved_egfr_similarity_category
novedad alta frente a aprobados    26
relacionado medicinalmente          3
análogo cercano de aprobado         1
Name: count, dtype: int64


## 14. Exportación de resultados

Se exportan los resultados siguiendo la estructura acordada:

- `outputs/scoring/intermediate/`: archivos intermedios auditables de cada etapa.
- `outputs/scoring/models/`: modelo predictivo de actividad y métricas.
- `outputs/scoring/final/`: archivos finales para el notebook 04.
- `outputs/scoring/reports/`: resumen de ejecución y criterios utilizados.

El archivo principal para el siguiente notebook es `outputs/scoring/final/shortlist30_final_candidates.csv`.

In [28]:
# Columnas prioritarias al inicio del CSV final.
cols = [
    "smiles", "predicted_pIC50", "reward_score",
    "admet_score", "admet_hard_pass_fraction",
    "max_tanimoto_training", "nearest_training_smiles",
    "max_tanimoto_approved_egfr", "nearest_approved_egfr_name", "approved_egfr_similarity_category",
    "qed", "ro5_violations", "mol_wt", "rdkit_logp", "tpsa", "hbd", "hba", "rot_bonds",
    "murcko_scaffold",
    "score_activity", "score_admet", "score_admet_hard", "score_training_similarity", "score_qed", "score_ro5",
]
admet_hard_pass_cols = [
    # Resultado booleano de cada filtro ADMET duro.
    "pass_PAINS_alert",
    "pass_NIH_alert",
    "pass_BRENK_alert",
    "pass_AMES",
    "pass_ClinTox",
    "pass_Carcinogens_Lagunin",
    "pass_hERG",
    "pass_Bioavailability_Ma",
    "pass_HIA_Hou",
]
basic_filter_cols = [
    # Resultado de los filtros fisicoquímicos básicos.
    "pass_mw_filter",
    "pass_logp_filter",
    "pass_tpsa_filter",
    "pass_hbd_filter",
    "pass_hba_filter",
    "pass_rot_bonds_filter",
    "pass_heavy_atoms_filter",
    "pass_ro5_filter",
    "pass_qed_filter",
    "n_basic_filter_failures",
    "pass_basic_chemical_filter",
]

priority_cols = cols + admet_hard_pass_cols + basic_filter_cols
df_shortlist_priority = df_shortlist[priority_cols]


In [29]:
# Se exporta el dataset completo con scores integrados para auditoría.
df.to_csv(SCORED_OUTPUT_PATH, index=False)

# Se exportan las moléculas que cumplen los umbrales mínimos definidos.
df_eligible.to_csv(ELIGIBLE_OUTPUT_PATH, index=False)

# Se exportan las candidatas pertenecientes al frente Pareto multiobjetivo.
df_pareto.to_csv(PARETO_OUTPUT_PATH, index=False)

# Se exporta una selección amplia y diversa para análisis exploratorio posterior.
df_top100.to_csv(TOP100_OUTPUT_PATH, index=False)

# Se exporta la shortlist final de 30 candidatas priorizadas.
df_shortlist_priority.to_csv(SHORTLIST_OUTPUT_PATH, index=False)


In [30]:

scoring_summary = {
    "created_at": datetime.now().isoformat(),
    "seed": SEED,
    "input_files": {
        "generated_molecules": str(GENERATED_PATH),
        "training_data": str(TRAINING_DATA_PATH),
        "reference_data": str(REFERENCE_DATA_PATH),
        "approved_egfr_reference": str(APPROVED_EGFR_REF_PATH),
    },
    "output_files": {
        "descriptors": str(DESCRIPTORS_OUTPUT_PATH),
        "filtered": str(FILTERED_OUTPUT_PATH),
        "admet_input": str(ADMET_INPUT_PATH),
        "admet_predictions": str(ADMET_OUTPUT_PATH),
        "scored": str(SCORED_OUTPUT_PATH),
        "eligible": str(ELIGIBLE_OUTPUT_PATH),
        "pareto": str(PARETO_OUTPUT_PATH),
        "top100": str(TOP100_OUTPUT_PATH),
        "shortlist30": str(SHORTLIST_OUTPUT_PATH),
        "activity_model": str(ACTIVITY_MODEL_PATH),
        "activity_metrics": str(ACTIVITY_METRICS_PATH),
    },
    "counts": {
        "generated_valid_unique": int(len(df_generated)),
        "filtered_basic_chemical": int(len(df_filtered)),
        "scored": int(len(df)),
        "eligible": int(len(df_eligible)),
        "pareto": int(len(df_pareto)),
        "top100": int(len(df_top100)),
        "shortlist30": int(len(df_shortlist)),
    },
    "basic_filters": BASIC_FILTERS,
    "eligibility_thresholds": ELIGIBILITY_THRESHOLDS,
    "pareto_objectives": pareto_objectives,
}

# Se guarda un resumen JSON para trazabilidad programática.
with open(SCORING_SUMMARY_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(scoring_summary, f, indent=4, ensure_ascii=False)

summary_lines = [
    "Resumen del notebook 03 — scoring y selección de candidatas EGFR",
    "=" * 68,
    f"Fecha de ejecución: {scoring_summary['created_at']}",
    "",
    "Conteos principales:",
    f"- Moléculas generadas válidas y únicas: {scoring_summary['counts']['generated_valid_unique']}",
    f"- Moléculas tras filtro químico básico: {scoring_summary['counts']['filtered_basic_chemical']}",
    f"- Moléculas con scoring completo: {scoring_summary['counts']['scored']}",
    f"- Moléculas elegibles: {scoring_summary['counts']['eligible']}",
    f"- Moléculas Pareto: {scoring_summary['counts']['pareto']}",
    f"- Top100 diverso: {scoring_summary['counts']['top100']}",
    f"- Shortlist final: {scoring_summary['counts']['shortlist30']}",
    "",
    "Archivo principal para el notebook 04:",
    f"- {SHORTLIST_OUTPUT_PATH}",
]

# Se guarda un resumen TXT para lectura rápida en la entrega.
with open(SCORING_SUMMARY_TXT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

print("Scored completo:", df.shape, "->", SCORED_OUTPUT_PATH)
print("Elegibles:", df_eligible.shape, "->", ELIGIBLE_OUTPUT_PATH)
print("Pareto:", df_pareto.shape, "->", PARETO_OUTPUT_PATH)
print("Top100 diverso:", df_top100.shape, "->", TOP100_OUTPUT_PATH)
print("Shortlist30 final:", df_shortlist.shape, "->", SHORTLIST_OUTPUT_PATH)
print("Resumen JSON:", SCORING_SUMMARY_JSON_PATH)
print("Resumen TXT:", SCORING_SUMMARY_TXT_PATH)


Scored completo: (8241, 144) -> /home/kluna/TFM/egfr_pipeline_tfm/outputs/scoring/intermediate/05_generated_molecules_scored.csv
Elegibles: (2133, 144) -> /home/kluna/TFM/egfr_pipeline_tfm/outputs/scoring/intermediate/06_eligible_candidates.csv
Pareto: (37, 144) -> /home/kluna/TFM/egfr_pipeline_tfm/outputs/scoring/intermediate/07_pareto_candidates.csv
Top100 diverso: (37, 144) -> /home/kluna/TFM/egfr_pipeline_tfm/outputs/scoring/final/top100_pareto_diverse.csv
Shortlist30 final: (30, 149) -> /home/kluna/TFM/egfr_pipeline_tfm/outputs/scoring/final/shortlist30_final_candidates.csv
Resumen JSON: /home/kluna/TFM/egfr_pipeline_tfm/outputs/scoring/reports/scoring_summary.json
Resumen TXT: /home/kluna/TFM/egfr_pipeline_tfm/outputs/scoring/reports/scoring_summary.txt


## 15. Conclusión metodológica

El pipeline final no selecciona moléculas únicamente por actividad predicha. Primero elimina estructuras químicamente poco plausibles, después estima actividad EGFR, controla la similitud frente al conjunto de entrenamiento y añade una capa ADMET para penalizar riesgos farmacocinéticos o toxicológicos.

La selección final se obtiene mediante un enfoque multiobjetivo: actividad, ADMET, filtros duros, similitud estructural, QED y cumplimiento de Lipinski. La aplicación posterior de Pareto y del límite por scaffold evita que el resultado sea una lista redundante de moléculas muy parecidas.

El resultado de este notebook es un CSV con las 30 candidatas priorizadas. Estas moléculas no deben presentarse como compuestos validados, sino como una shortlist computacional defendible para el siguiente análisis.
